# Alibaba Cloud Serverless Performance Analysis
**Course Project | Cloud Computing**  
*Dataset: Real Alibaba Cloud Function Compute invocation logs - 21 serverless functions*

In [ ]:
!git clone https://github.com/tahaglbz/Alibaba-Serverless-Analysis.git
%cd Alibaba-Serverless-Analysis
!pip install pandas openpyxl xlrd matplotlib seaborn scikit-learn scipy xgboost -q
!mkdir -p analysis_output
!python preprocess.py
%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
print(f'All imports successful.')

## 1. Data Loading & Overview

In [ ]:
df_logs = pd.read_csv('data/processed/functions_combined.csv')
df_perf = pd.read_csv('data/processed/perf_profile_combined.csv')

# Keep full df for success rate calc, filter for analysis
df_full = df_logs.copy()
df = df_logs[df_logs['state'] == 'Success'].copy().reset_index(drop=True)

print(f"Invoke logs (all):     {df_full.shape[0]:,} rows")
print(f"Invoke logs (Success): {df.shape[0]:,} rows")
print(f"Perf profiles:         {df_perf.shape[0]:,} rows")
print(f"Unique functions:      {df['function_id'].nunique()}")
print(f"Function sets:         {sorted(df['function_set'].unique())}")
print(f"Success rate:          {(df_full['state'] == 'Success').mean() * 100:.1f}%")
print()
display(df.describe().round(2))

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-left: Avg duration per function, color-coded by tier
func_stats = df.groupby('function_id')['billed_duration_ms'].mean().sort_values(ascending=False)
colors = ['#e74c3c' if v > 5000 else '#f39c12' if v > 3000 else '#2ecc71' for v in func_stats.values]
bars = axes[0, 0].bar(func_stats.index, func_stats.values, color=colors, edgecolor='white')
axes[0, 0].set_title('Avg Billed Duration by Function', fontweight='bold')
axes[0, 0].set_xlabel('Function ID')
axes[0, 0].set_ylabel('Avg Duration (ms)')
axes[0, 0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, func_stats.values):
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                    f'{val:.0f}', ha='center', va='bottom', fontsize=7)
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Fast (<3000ms)'),
    mpatches.Patch(color='#f39c12', label='Medium (3000-5000ms)'),
    mpatches.Patch(color='#e74c3c', label='Slow (>5000ms)')
]
axes[0, 0].legend(handles=legend_patches, fontsize=8)

# Top-right: Duration distribution with KDE
axes[0, 1].hist(df['billed_duration_ms'], bins=60, color='#3498db', alpha=0.6,
                edgecolor='white', density=True, label='Histogram')
df['billed_duration_ms'].plot.kde(ax=axes[0, 1], color='#e74c3c', linewidth=2, label='KDE')
axes[0, 1].set_title('Billed Duration Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Duration (ms)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Bottom-left: Memory vs Duration scatter + regression
axes[1, 0].scatter(df['memory_mb'], df['billed_duration_ms'], alpha=0.15, s=10, color='#9b59b6')
m, b, r, p, _ = stats.linregress(df['memory_mb'], df['billed_duration_ms'])
x_line = np.linspace(df['memory_mb'].min(), df['memory_mb'].max(), 100)
axes[1, 0].plot(x_line, m * x_line + b, color='#e74c3c', linewidth=2,
                label=f'Regression (r={r:.3f})')
axes[1, 0].set_title('Memory vs Duration', fontweight='bold')
axes[1, 0].set_xlabel('Memory (MB)')
axes[1, 0].set_ylabel('Duration (ms)')
axes[1, 0].legend()

# Bottom-right: Duration by function_set
df.boxplot(column='billed_duration_ms', by='function_set',
           ax=axes[1, 1], showfliers=False,
           boxprops=dict(color='#2c3e50'),
           medianprops=dict(color='#e74c3c', linewidth=2))
axes[1, 1].set_title('Duration by Function Set', fontweight='bold')
axes[1, 1].set_xlabel('Function Set')
axes[1, 1].set_ylabel('Duration (ms)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('analysis_output/eda.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Memory-Duration Pearson r = {r:.4f} (will revisit in statistical analysis)")

## 3. Statistical Analysis

In [ ]:
print(f"{'=' * 55}")
print(f"CORRELATION ANALYSIS: Memory vs Billed Duration")
print(f"{'=' * 55}")

pearson_r, pearson_p = stats.pearsonr(df['memory_mb'], df['billed_duration_ms'])
spearman_r, spearman_p = stats.spearmanr(df['memory_mb'], df['billed_duration_ms'])

direction = 'POSITIVE' if pearson_r > 0 else 'NEGATIVE'
sig = 'significant' if pearson_p < 0.05 else 'not significant'
print(f"Pearson  r = {pearson_r:+.4f}  p = {pearson_p:.4e}  -> {direction}, {sig}")
print(f"Spearman rho = {spearman_r:+.4f}  p = {spearman_p:.4e}")

if pearson_r > 0.05 and pearson_p < 0.05:
    print(f"\n[ALERT] MEMORY PARADOX CONFIRMED: Higher memory -> LONGER duration")
    print(f"  This counter-intuitive result suggests memory allocation overhead")
    print(f"  or cold-start behavior outweighs compute benefits for these functions.")
elif pearson_r < -0.05 and pearson_p < 0.05:
    print(f"\n[OK] Expected behavior: Higher memory -> SHORTER duration")
else:
    print(f"\nNo significant linear relationship between memory and duration")

print()
print(f"{'=' * 55}")
print(f"ANOVA: Duration differences across functions")
print(f"{'=' * 55}")
groups = [g['billed_duration_ms'].values for _, g in df.groupby('function_id')]
f_stat, anova_p = stats.f_oneway(*groups)
print(f"F-statistic = {f_stat:.2f}  p = {anova_p:.4e}")
if anova_p < 0.05:
    print(f"-> Statistically significant differences exist between functions (p < 0.05)")

print()
print(f"{'=' * 55}")
print(f"TOP 5 FASTEST FUNCTIONS (by median duration)")
print(f"{'=' * 55}")
fastest = df.groupby('function_id')['billed_duration_ms'].median().nsmallest(5)
for fn, val in fastest.items():
    print(f"  {fn}: {val:.0f} ms")

print()
print(f"TOP 5 MOST VARIABLE FUNCTIONS (by std deviation)")
print(f"{'=' * 55}")
most_variable = df.groupby('function_id')['billed_duration_ms'].std().nlargest(5)
for fn, val in most_variable.items():
    print(f"  {fn}: sigma = {val:.0f} ms")

## 4. CPU & Memory Performance Profile Analysis

In [ ]:
# Bin memory and cpu into ranges for heatmap
df_perf['memory_bin'] = pd.cut(df_perf['memory_mb'],
    bins=[0, 512, 1024, 1536, 2048, 2560, 3072, 3584, 4096],
    labels=['<=512', '513-1024', '1025-1536', '1537-2048',
            '2049-2560', '2561-3072', '3073-3584', '3585-4096'])
df_perf['cpu_bin'] = pd.cut(df_perf['cpu_cores'],
    bins=[0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0],
    labels=['<=0.5', '0.6-1.0', '1.1-1.5', '1.6-2.0',
            '2.1-2.5', '2.6-3.0', '3.1-3.5', '3.6-4.0'])

pivot = df_perf.groupby(['memory_bin', 'cpu_bin'])['duration_ms'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, ax=ax, cmap='RdYlGn_r', annot=True, fmt='.0f',
            linewidths=0.3, cbar_kws={'label': 'Avg Duration (ms)'})
ax.set_title('Avg Duration Heatmap: Memory x CPU Config (Perf Profile)', fontweight='bold')
ax.set_xlabel('CPU Cores Range')
ax.set_ylabel('Memory (MB) Range')
plt.tight_layout()
plt.savefig('analysis_output/perf_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Machine Learning: Multi-Model Performance Classification

We train and compare **5 ML/DL models** to classify each serverless function invocation
as **High Performance** (billed duration below median) or **Low Performance**
(billed duration at or above median).

**Models selected for this dataset:**
- Logistic Regression: linear baseline, highly interpretable
- Random Forest: handles non-linear patterns, robust with small feature sets
- Gradient Boosting: typically best accuracy on tabular data with few features
- SVM (RBF kernel): effective in 2D feature space, strong decision boundary
- MLP Neural Network: satisfies deep learning requirement, captures complex interactions

In [ ]:
threshold = df['billed_duration_ms'].median()
y = (df['billed_duration_ms'] < threshold).astype(int)

print(f"Classification threshold (median): {threshold:.0f} ms")
print(f"High Performance (1): {y.sum():,}  ({y.mean() * 100:.1f}%)")
print(f"Low Performance  (0): {(1 - y).sum():,}  ({(1 - y.mean()) * 100:.1f}%)")

X = df[['memory_mb', 'cpu_cores']].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"\nTrain: {X_train_s.shape[0]:,} samples | Test: {X_test_s.shape[0]:,} samples")

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                   max_depth=4, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale',
                     probability=True, random_state=42),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32),
                                        activation='relu', max_iter=500,
                                        early_stopping=True, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    print(f"Trained: {name}")

print()
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>7} {'ROC-AUC':>9}")
print(f"{'-' * 70}")
sorted_models = sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True)
for name, r in sorted_models:
    print(f"{name:<22} {r['accuracy']:>9.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>7.4f} {r['roc_auc']:>9.4f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low', 'High'],
                yticklabels=['Low', 'High'],
                linewidths=0.5, cbar=False)
    acc = r['accuracy']
    ax.set_title(f"{name}\nAcc={acc:.3f}", fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual', fontsize=8)

plt.suptitle('Confusion Matrices - All 5 Models', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('analysis_output/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#f39c12']

for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name} (AUC = {r['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline (AUC = 0.500)')
ax.set_title('ROC Curves - All 5 Models', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analysis_output/roc_curves.png', dpi=120)
plt.show()

In [ ]:
explanations = {
    'Logistic Regression': (
        'Fits a linear decision boundary by estimating the probability of each class '
        'using the sigmoid function and log-loss optimization.',
        'Serves as an interpretable baseline. Effective when the relationship between '
        'memory/CPU and performance is approximately linear. Low computational cost.'
    ),
    'Random Forest': (
        'Builds an ensemble of decision trees using bootstrap sampling and random '
        'feature selection, averaging predictions to reduce variance.',
        'Well-suited for this dataset: robust against noise, provides feature importance, '
        'and handles non-linear memory-duration behavior without feature engineering.'
    ),
    'Gradient Boosting': (
        'Sequentially trains shallow trees where each tree corrects the residual errors '
        'of the previous one, optimizing a differentiable loss function.',
        'Often the strongest performer on small-feature tabular data. Captures subtle '
        'threshold effects in memory/CPU configurations that Random Forest may miss.'
    ),
    'SVM (RBF)': (
        'Finds the maximum-margin hyperplane separating classes, using the RBF (Gaussian) '
        'kernel to map features into a higher-dimensional space implicitly.',
        'Particularly effective in low-dimensional feature spaces (2 features here). '
        'The RBF kernel captures non-linear boundaries between performance tiers.'
    ),
    'MLP Neural Network': (
        'A feedforward neural network with two hidden layers (64->32 neurons, ReLU activation) '
        'trained via backpropagation to learn hierarchical feature representations.',
        'Satisfies the deep learning component requirement. With sufficient data it can '
        'approximate complex non-linear decision boundaries beyond kernel methods.'
    )
}

print(f"{'=' * 65}")
print(f"MODEL EXPLANATIONS & SUITABILITY FOR THIS DATASET")
print(f"{'=' * 65}")
for i, (name, (how, why)) in enumerate(explanations.items(), 1):
    r = results[name]
    print(f"\n{'-' * 65}")
    print(f"  {i}. {name}")
    print(f"{'-' * 65}")
    print(f"  How it works: {how}")
    print(f"  Suitability:  {why}")
    print(f"  Results:      Accuracy={r['accuracy']:.4f} | Precision={r['precision']:.4f} "
          f"| Recall={r['recall']:.4f} | F1={r['f1']:.4f} | ROC-AUC={r['roc_auc']:.4f}")
print(f"\n{'=' * 65}")

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best_model = results[best_name]['model']
print(f"Best model by F1-Score: {best_name}  (F1 = {results[best_name]['f1']:.4f})")

fig, ax = plt.subplots(figsize=(6, 4))
feature_names = ['Memory (MB)', 'CPU Cores']
importances = None

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    ax.barh(feature_names, importances, color=['#3498db', '#e74c3c'], edgecolor='white')
    ax.set_title(f"Feature Importances - {best_name}", fontweight='bold')
    ax.set_xlabel('Importance Score')
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
    ax.barh(feature_names, importances, color=['#3498db', '#e74c3c'], edgecolor='white')
    ax.set_title(f"Feature Coefficients (abs) - {best_name}", fontweight='bold')
    ax.set_xlabel('|Coefficient|')
else:
    ax.text(0.5, 0.5, f"{best_name} does not expose\nfeature importances directly",
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title(f"Feature Importance - {best_name}", fontweight='bold')

if importances is not None:
    for i, v in enumerate(importances):
        ax.text(v + 0.001, i, f"{v:.4f}", va='center', fontsize=10)

plt.tight_layout()
plt.savefig('analysis_output/feature_importance.png', dpi=120)
plt.show()

## 6. Function Performance Grading

In [ ]:
grade_stats = df.groupby('function_id').agg(
    mean_duration=('billed_duration_ms', 'mean'),
    median_duration=('billed_duration_ms', 'median'),
    std_duration=('billed_duration_ms', 'std'),
    sample_count=('billed_duration_ms', 'count')
).reset_index()

# Success rate from full df
sr = df_full.groupby('function_id').apply(
    lambda x: (x['state'] == 'Success').mean() * 100
).reset_index(name='success_rate')
grade_stats = grade_stats.merge(sr, on='function_id', how='left')

# Assign grades based on mean_duration percentile
pct = grade_stats['mean_duration'].rank(pct=True)
grade_stats['grade'] = pd.cut(
    pct,
    bins=[0, 0.10, 0.30, 0.70, 0.90, 1.01],
    labels=['A', 'B', 'C', 'D', 'F']
)

grade_stats = grade_stats.sort_values('mean_duration')
print(f"FUNCTION PERFORMANCE GRADES")
print(f"{'=' * 75}")
print(f"{'Function':<12} {'Grade':>6} {'Mean(ms)':>10} {'Median(ms)':>12} {'Std(ms)':>9} {'Success%':>10} {'N':>7}")
print(f"{'-' * 75}")
for _, row in grade_stats.iterrows():
    print(f"{row['function_id']:<12} {str(row['grade']):>6} {row['mean_duration']:>10.0f} "
          f"{row['median_duration']:>12.0f} {row['std_duration']:>9.0f} {row['success_rate']:>9.1f}% "
          f"{row['sample_count']:>7,}")

worst = grade_stats.iloc[-1]
print(f"\nWorst function: {worst['function_id']}  Mean={worst['mean_duration']:.0f}ms  "
      f"Std={worst['std_duration']:.0f}ms  Grade={worst['grade']}")

## 7. Summary & Conclusions

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['f1'])
best_f1 = results[best_model_name]['f1']
best_auc = results[best_model_name]['roc_auc']
fastest_fn = grade_stats.iloc[0]
slowest_fn = grade_stats.iloc[-1]

print(f"{'=' * 60}")
print(f"  ALIBABA SERVERLESS PERFORMANCE ANALYSIS - SUMMARY")
print(f"{'=' * 60}")
print(f"\n  DATASET")
print(f"  - Total invocations (Success): {df.shape[0]:,}")
print(f"  - Unique functions:            {df['function_id'].nunique()}")
print(f"  - Function sets:               NEW10, NEW16, NEW21")
print(f"  - Overall success rate:        {(df_full['state'] == 'Success').mean() * 100:.1f}%")

print(f"\n  KEY FINDINGS")
direction_str = 'POSITIVE (Memory Paradox)' if pearson_r > 0 else 'NEGATIVE (expected)'
print(f"  - Memory-Duration correlation: r={pearson_r:+.4f} -> {direction_str}")
print(f"  - Fastest function:  {fastest_fn['function_id']}  ({fastest_fn['mean_duration']:.0f} ms avg)")
print(f"  - Slowest function:  {slowest_fn['function_id']}  ({slowest_fn['mean_duration']:.0f} ms avg)")

print(f"\n  ML RESULTS (5 models trained & compared)")
for name, r in sorted_models:
    marker = '*' if name == best_model_name else ' '
    print(f"  {marker} {name:<22} F1={r['f1']:.4f}  AUC={r['roc_auc']:.4f}")

print(f"\n  Best model: {best_model_name}  (F1={best_f1:.4f}, ROC-AUC={best_auc:.4f})")

print(f"\n  OPTIMIZATION RECOMMENDATIONS")
print(f"  1. Reduce memory allocation for functions showing positive memory-duration")
print(f"     correlation to improve resource efficiency and reduce latency overhead.")
print(f"  2. Investigate {slowest_fn['function_id']} (Grade F): high mean and variance suggest")
print(f"     cold-start issues or inefficient resource utilization.")
print(f"  3. Use the {best_model_name} model to predict performance before deployment,")
print(f"     enabling proactive resource right-sizing.")
print(f"{'=' * 60}")